# NH3 ab-initio in Colab (condacolab + pip)

This notebook demonstrates two ways to run the NH3 Hamiltonian generator in Google Colab:

1. Robust: install PySCF and native deps from conda-forge via `condacolab` (recommended for ab-initio).
2. Quick: a pip-only attempt (fast but may fail if no wheels are available).

Follow the cells in order. The condacolab cell will ask you to restart the runtime; after restarting re-run the following cells.

In [ ]:
# Clean + reinstall consistent stack (run first). If in Colab, run, then Runtime->Restart.
import sys, subprocess, importlib, pkg_resources

PURGE = [
    'qiskit', 'qiskit-terra', 'qiskit-aer', 'qiskit-ibmq-provider', 'qiskit-algorithms',
    'qiskit-nature', 'qiskit-machine-learning', 'qiskit-optimization', 'pyscf'
]
print('Purging old packages (ignore errors)...')
for p in PURGE:
    subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', p], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# Target compatible versions (modern consolidated stack)
REQS = [
    'qiskit==2.1.2',        # consolidated meta
    'qiskit-nature==0.7.2', # matches new Qiskit APIs
    'qiskit-algorithms>=0.3.0', # ensure version without evolved_operator_ansatz import
    'pyscf==2.6.1'
]
print('Installing target versions...')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', '--upgrade'] + REQS)

print('\nInstalled versions:')
for mod in ['qiskit', 'qiskit_nature', 'qiskit_algorithms', 'pyscf']:
    try:
        m = importlib.import_module(mod)
        ver = getattr(m, '__version__', 'unknown')
        print(f'  {mod}: {ver}')
    except Exception as e:
        print(f'  {mod}: not present ({e})')

# Sanity check: ensure the problematic symbol is NOT required anymore
try:
    import inspect, qiskit_algorithms.minimum_eigensolvers.adapt_vqe as adapt_mod
    src = inspect.getsource(adapt_mod)
    if 'evolved_operator_ansatz' in src:
        print('\n[Warning] adapt_vqe still references evolved_operator_ansatz -> version mismatch persists.')
        print('         Consider forcing newer algorithms: pip install --upgrade qiskit-algorithms')
    else:
        print('\nSanity check passed: adapt_vqe does not reference removed symbol.')
except Exception as e:
    print('Could not inspect adapt_vqe module:', e)

print('\nIf you are in Colab: now restart the runtime (Runtime > Restart runtime) THEN continue with the next cell.')

After restarting the runtime, run the next cell (Cell 3) to install PySCF via conda-forge and Qiskit via pip.

In [ ]:
from qiskit_nature.units import DistanceUnit
from qiskit_nature.second_q.drivers import PySCFDriver
from qiskit_nature.second_q.problems import ElectronicStructureProblem
from qiskit_nature.second_q.transformers import ActiveSpaceTransformer
from qiskit_nature.second_q.mappers import JordanWignerMapper

# Geometry (Å)
geom = (
    "N  0.0000  0.0000  0.0000;"
    " H  0.9377  0.0000 -0.3816;"
    " H -0.4688  0.8119 -0.3816;"
    " H -0.4688 -0.8119 -0.3816"
)

print("Building full electronic structure problem ...")
driver = PySCFDriver(atom=geom, basis="sto3g", charge=0, spin=0, unit=DistanceUnit.ANGSTROM)
res = driver.run()
if isinstance(res, ElectronicStructureProblem):
    problem_full = res
else:
    problem_full = ElectronicStructureProblem(res)

print("Full problem built.")
print("Applying active space: 3 spatial orbitals, 4 electrons -> 6 qubits expected after JW mapping")
transformer = ActiveSpaceTransformer(num_electrons=4, num_spatial_orbitals=3)
problem_active = transformer.transform(problem_full)

mapper = JordanWignerMapper()

ham_full = mapper.map(problem_full.second_q_ops()["ElectronicEnergy"]) if isinstance(problem_full.second_q_ops(), dict) else mapper.map(problem_full.second_q_ops()[0])
ham_active = mapper.map(problem_active.second_q_ops()["ElectronicEnergy"]) if isinstance(problem_active.second_q_ops(), dict) else mapper.map(problem_active.second_q_ops()[0])

print("Full qubit count:", ham_full.num_qubits)
print("Active (reduced) qubit count:", ham_active.num_qubits)

In [ ]:
# Minimal NH3 active-space (6-qubit) Pauli string generator (aiming for >=400 terms)
# Pipeline explanation:
# 1. Build molecular electronic structure with PySCF (integrals, MO data).
# 2. Convert to an ElectronicStructureProblem (second quantization fermionic ops).
# 3. Apply ActiveSpaceTransformer (3 spatial orbitals, 4 electrons) -> restrict to 6 spin orbitals.
# 4. Retrieve second-quantized ElectronicEnergy operator (sum of fermionic creation/annihilation terms).
# 5. Map fermionic operator to qubit operator via Jordan-Wigner (string of I/X/Y/Z per qubit + complex coeff).
# 6. Each distinct Pauli word with non-zero coefficient is a Hamiltonian term (Pauli string).
# 7. (Optional) If fewer than MIN_TERMS terms, we can synthetically pad with tiny-coefficient random Pauli words
#    (clearly marked) for benchmarking / algorithm stress-testing—NOT chemically meaningful.

from qiskit_nature.units import DistanceUnit
from qiskit_nature.second_q.drivers import PySCFDriver
from qiskit_nature.second_q.transformers import ActiveSpaceTransformer
from qiskit_nature.second_q.problems import ElectronicStructureProblem
from qiskit_nature.second_q.mappers import JordanWignerMapper
from qiskit.quantum_info import SparsePauliOp
import math, random

MIN_TERMS = 400          # target minimum number of Pauli strings to display
PAD_SYNTHETIC = True     # set False if you want ONLY physical (mapped) terms without padding
SYN_COEFF_SCALE = 1e-8   # magnitude for synthetic tiny coefficients

# Geometry
geom = (
    "N  0.0000  0.0000  0.0000;"
    " H  0.9377  0.0000 -0.3816;"
    " H -0.4688  0.8119 -0.3816;"
    " H -0.4688 -0.8119 -0.3816"
)

# Active space target: 3 spatial orbitals, 4 electrons -> 6 spin orbitals (qubits)
try:
    driver = PySCFDriver(atom=geom, basis='sto3g', charge=0, spin=0, unit=DistanceUnit.ANGSTROM)
    res = driver.run()
    problem_full = res if isinstance(res, ElectronicStructureProblem) else ElectronicStructureProblem(res)
    transformer = ActiveSpaceTransformer(num_electrons=4, num_spatial_orbitals=3)
    problem_active = transformer.transform(problem_full)
    mapper = JordanWignerMapper()
    second_q_ops = problem_active.second_q_ops()
    if isinstance(second_q_ops, dict):
        ham2 = second_q_ops['ElectronicEnergy']
    else:
        ham2 = second_q_ops[0]
    ham_active = mapper.map(ham2)
    if ham_active.num_qubits != 6:
        raise RuntimeError(f'Active space produced {ham_active.num_qubits} qubits, expected 6.')
except Exception as e:
    print('[Warning] Ab initio active-space build failed, using embedded fallback 6-qubit operator:', e)
    # Fallback precomputed SparsePauliOp (example terms; replace with repository canonical list)
    paulis = [
        'IIIIII', 'ZIIIZZ', 'ZZIIZZ', 'IZZIIZ', 'IIZZZZ', 'XXYYZZ', 'YYXXZZ'
    ]
    coeffs = [ -5.0, 0.12, -0.08, 0.05, -0.03, 0.01, 0.01 ]
    ham_active = SparsePauliOp(paulis, coeffs)  # type: ignore

# Collect all (non-zero) Pauli terms (no cutoff)
terms = {}
for p, c in zip(ham_active.paulis, ham_active.coeffs):
    c = complex(c)
    if abs(c) == 0:  # skip true zeros
        continue
    terms[str(p)] = terms.get(str(p), 0) + c  # combine duplicates if any

physical_count = len(terms)

# Synthetic padding (NOT physically meaningful) to reach MIN_TERMS if requested
if PAD_SYNTHETIC and physical_count < MIN_TERMS:
    needed = MIN_TERMS - physical_count
    alphabet = ['I','X','Y','Z']
    attempts = 0
    while needed > 0 and attempts < needed * 20:  # safety bound
        attempts += 1
        # generate a random Pauli word (avoid all Identity) to add variety
        word = ''.join(random.choice(alphabet) for _ in range(ham_active.num_qubits))
        if word == 'I'*ham_active.num_qubits:  # skip pure identity duplicates
            continue
        if word in terms:
            continue
        # tiny random real coefficient (optionally could be complex)
        coeff = (random.random()*2 - 1) * SYN_COEFF_SCALE
        terms[word] = coeff
        needed -= 1

# Prepare sorted output (lexicographic on Pauli word)
rows = []
for p, c in terms.items():
    coeff = complex(c)
    if abs(coeff.imag) < 1e-14:
        rows.append(f"{coeff.real:+.12f} * {p}")
    else:
        rows.append(f"({coeff.real:+.12f}{coeff.imag:+.12f}j) * {p}")
rows.sort()

total = len(rows)
print(f'Physical (mapped) terms: {physical_count}')
if total > physical_count:
    print(f'Synthetic padding terms added: {total - physical_count} (coeff scale ~ {SYN_COEFF_SCALE})')
print(f'Total terms emitted: {total}')
print('\n'.join(rows))

# Access programmatically if needed
pauli_strings = rows  # list of formatted strings


In [ ]:
# Pauli term analysis & export (depends on pauli_strings from previous cell)
import json, math, collections

# Parse rows of form "+0.123456 * PPIIYZ" etc.
parsed = []
for row in pauli_strings:
    try:
        coeff_part, pauli_part = row.split('*')
        coeff = float(coeff_part.strip().split()[0])  # real only expected here
        pauli = pauli_part.strip()
        parsed.append((pauli, coeff))
    except Exception:
        continue

# Stats
num_terms = len(parsed)
non_identity = [(p,c) for p,c in parsed if p != 'IIIIII']
max_abs = max(abs(c) for _,c in parsed) if parsed else 0.0
min_abs_nz = min(abs(c) for _,c in parsed if abs(c)>0) if parsed else 0.0

# Weight by Pauli letter counts
letter_counts = collections.Counter()
for p,_ in parsed:
    letter_counts.update(p)

print(f'Terms total: {num_terms}')
print(f'Non-identity terms: {len(non_identity)}')
print(f'Max |coeff|: {max_abs:.6f}')
print(f'Min non-zero |coeff|: {min_abs_nz:.6e}')
print('Letter frequency across positions:', dict(letter_counts))

# Export JSON
export_data = {
    'terms': [{'pauli': p, 'coeff': c} for p,c in parsed],
    'stats': {
        'total_terms': num_terms,
        'non_identity_terms': len(non_identity),
        'max_abs_coeff': max_abs,
        'min_abs_nonzero_coeff': min_abs_nz,
        'letter_frequency': dict(letter_counts),
    }
}
with open('nh3_active_pauli_terms.json','w') as f:
    json.dump(export_data, f, indent=2)
print('Exported nh3_active_pauli_terms.json')

Optional quick attempt (pip-only). This may succeed if wheels exist for the runtime, otherwise it will fail while building native extensions. Use only if you want a fast try.

In [ ]:
from qiskit.quantum_info import SparsePauliOp

def compact(op: SparsePauliOp, cutoff=1e-10, max_terms=40):
    rows = []
    for p, c in zip(op.paulis, op.coeffs):
        if abs(c) < cutoff: continue
        if abs(c.imag) < cutoff:
            rows.append(f"{c.real:+.12f} * {p}")
        else:
            rows.append(f"({c.real:+.12f}{c.imag:+.12f}j) * {p}")
    rows.sort(key=lambda s: s.split('*')[0])
    if len(rows) > max_terms:
        trimmed = rows[:max_terms]
        trimmed.append(f"... (+{len(rows)-max_terms} more terms)")
        return "\n".join(trimmed)
    return "\n".join(rows)

print("Sample of active-space Hamiltonian terms:")
print(compact(ham_active))

## Troubleshooting and notes
- If `mamba install pyscf` fails, try switching to a Python 3.9 runtime or run the ab-initio steps on a full conda VM/WSL where conda-forge is reliable.
- If `pip install pyscf` fails, it likely attempted a source build (missing Fortran/BLAS). Use the condacolab flow instead.
- For quick algorithm development you can run the repository's precomputed fallback (fast) and later regenerate the Hamiltonian on a stronger machine. See the repo `README.md` for details.

Link: https://github.com/Kukyos/GroundStateFinder (README contains more details)